# Similar Movies with Collaborative Filtering

This notebook uses MovieLens 100K ratings to find movies with similar user-rating patterns. It compares basic Pearson correlation with an overlap-aware shrinkage score.

In [11]:
# 1. Import Required Libraries
import pandas as pd
import numpy as np

In [12]:
# 2. Load MovieLens Datasets
from pathlib import Path

DATA_DIR = Path("/Users/partha/AI_ML_Course/MLCourse/ml-100k")
ratings_path = DATA_DIR / "u.data"
movies_path = DATA_DIR / "u.item"

if not ratings_path.exists() or not movies_path.exists():
    raise FileNotFoundError(
        f"MovieLens files were not found in {DATA_DIR}. Update DATA_DIR to your dataset location."
    )

rating_data = pd.read_csv(
    ratings_path,
    sep="\t",
    names=["user_id", "movie_id", "rating", "timestamp"],
    encoding="ISO-8859-1",
)

movies = pd.read_csv(
    movies_path,
    sep="|",
    names=["movie_id", "title"],
    usecols=[0, 1],
    encoding="ISO-8859-1",
)

rating_data.head()

,user_id,movie_id,rating,timestamp
0,0,50,5,881250949
1,0,172,5,881250949
2,0,133,1,881250949
3,196,242,3,881250949
4,186,302,3,891717742


In [13]:
# 3. Merge Ratings and Movie Titles
ratings = rating_data.merge(movies, on="movie_id", how="inner")
ratings[["user_id", "movie_id", "title", "rating"]].head()

,user_id,movie_id,title,rating
0,0,50,Star Wars (1977),5
1,0,172,"Empire Strikes Back, The (1980)",5
2,0,133,Gone with the Wind (1939),1
3,196,242,Kolya (1996),3
4,186,302,L.A. Confidential (1997),3


In [14]:
# 4. Create User-Movie Ratings Pivot Table
movie_ratings = ratings.pivot_table(
    index="user_id",
    columns="title",
    values="rating",
)

movie_ratings.shape

(944, 1664)

In [15]:
# 5. Compute Movie Statistics
movie_stats = ratings.groupby("title")["rating"].agg(
    rating_count="count",
    average_rating="mean",
)

movie_stats.sort_values("rating_count", ascending=False).head()

,rating_count,average_rating
title,,
Star Wars (1977),584,4.359589
Contact (1997),509,3.803536
Fargo (1996),508,4.155512
Return of the Jedi (1983),507,4.007890
Liar Liar (1997),485,3.156701


In [16]:
# 6. Filter Popular Movies
MIN_RATINGS = 250
popular_movies = movie_stats.loc[movie_stats["rating_count"] >= MIN_RATINGS]

popular_movies.sort_values("average_rating", ascending=False).head(10)

,rating_count,average_rating
title,,
Schindler's List (1993),298,4.466443
"Shawshank Redemption, The (1994)",283,4.445230
"Usual Suspects, The (1995)",267,4.385768
Star Wars (1977),584,4.359589
One Flew Over the Cuckoo's Nest (1975),264,4.291667
"Silence of the Lambs, The (1991)",390,4.289744
"Godfather, The (1972)",413,4.283293
Raiders of the Lost Ark (1981),420,4.252381
Titanic (1997),350,4.245714


In [22]:
# 7. Find Similar Movies Using Correlation
def pairwise_correlations(target_ratings, candidate_titles):
    correlations = {}

    for candidate_title in candidate_titles:
        paired_ratings = pd.concat(
            [target_ratings, movie_ratings[candidate_title]],
            axis=1,
        ).dropna()
        if len(paired_ratings) < 2:
            continue
        if paired_ratings.iloc[:, 0].nunique() < 2 or paired_ratings.iloc[:, 1].nunique() < 2:
            continue
        correlations[candidate_title] = paired_ratings.iloc[:, 0].corr(
            paired_ratings.iloc[:, 1]
        )

    return pd.Series(correlations, dtype=float)


TARGET_TITLE = "Batman Returns (1992)"

if TARGET_TITLE not in movie_ratings.columns:
    raise KeyError(f"Movie not found in the ratings data: {TARGET_TITLE}")

target_ratings = movie_ratings[TARGET_TITLE]
common_users = movie_ratings.notna().T.dot(target_ratings.notna().astype(int))
eligible_titles = common_users.index[common_users >= 2]
correlations = pairwise_correlations(target_ratings, eligible_titles)

basic_results = pd.DataFrame({
    "correlation": correlations,
    "common_users": common_users,
    "rating_count": movie_stats["rating_count"],
})
basic_results = basic_results.drop(index=TARGET_TITLE, errors="ignore")
basic_results = basic_results.loc[
    (basic_results["rating_count"] >= MIN_RATINGS)
    & (basic_results["common_users"] >= 2)
]
basic_results.sort_values("correlation", ascending=False).head(10)

,correlation,common_users,rating_count
Indiana Jones and the Last Crusade (1989),0.344773,120,331
Mr. Holland's Opus (1995),0.322247,59,293
Schindler's List (1993),0.314059,86,298
"Silence of the Lambs, The (1991)",0.302875,109,390
One Flew Over the Cuckoo's Nest (1975),0.290601,76,264
Jurassic Park (1993),0.286174,120,261
Back to the Future (1985),0.285798,120,350
Dead Man Walking (1995),0.275720,59,299
Pulp Fiction (1994),0.262129,121,394
Terminator 2: Judgment Day (1991),0.261105,114,295


In [23]:
# 8. Define Function to Find Similar Movies
def find_similar_movies(movie_title, n=10, min_movie_ratings=250):
    if movie_title not in movie_ratings.columns:
        raise KeyError(f"Movie not found in the ratings data: {movie_title}")

    target_ratings = movie_ratings[movie_title]
    common_users = movie_ratings.notna().T.dot(target_ratings.notna().astype(int))
    candidate_titles = movie_stats.index[
        (movie_stats["rating_count"] >= min_movie_ratings)
        & (common_users >= 2)
    ]
    candidate_titles = candidate_titles.drop(movie_title, errors="ignore")
    correlations = pairwise_correlations(target_ratings, candidate_titles)

    results = pd.DataFrame({
        "correlation": correlations,
        "common_users": common_users,
        "rating_count": movie_stats["rating_count"],
    })
    return results.sort_values("correlation", ascending=False).head(n)

In [24]:
# 9. Advanced Similarity Function with Shrinkage
def find_similar_movies_shrunk(
    movie_title,
    n=10,
    min_common_users=20,
    min_movie_ratings=100,
    shrinkage=50,
):
    if movie_title not in movie_ratings.columns:
        raise KeyError(f"Movie not found in the ratings data: {movie_title}")
    if shrinkage < 0:
        raise ValueError("shrinkage must be non-negative")

    target_ratings = movie_ratings[movie_title]
    common_users = movie_ratings.notna().T.dot(target_ratings.notna().astype(int))
    candidate_titles = movie_stats.index[
        (movie_stats["rating_count"] >= min_movie_ratings)
        & (common_users >= min_common_users)
    ]
    candidate_titles = candidate_titles.drop(movie_title, errors="ignore")
    correlations = pairwise_correlations(target_ratings, candidate_titles)

    results = pd.DataFrame({
        "correlation": correlations,
        "common_users": common_users,
        "rating_count": movie_stats["rating_count"],
    }).dropna(subset=["correlation"])

    results["score"] = (
        results["correlation"]
        * results["common_users"]
        / (results["common_users"] + shrinkage)
    )

    return results.sort_values("score", ascending=False).head(n)

In [25]:
# 10. Test Similarity Functions
print("Basic Pearson-correlation recommendations:")
display(find_similar_movies("Batman Returns (1992)", n=10))

print("Overlap-adjusted recommendations:")
display(find_similar_movies_shrunk("Batman Returns (1992)", n=10))

Basic Pearson-correlation recommendations:


,correlation,common_users,rating_count
Indiana Jones and the Last Crusade (1989),0.344773,120,331
Mr. Holland's Opus (1995),0.322247,59,293
Schindler's List (1993),0.314059,86,298
"Silence of the Lambs, The (1991)",0.302875,109,390
One Flew Over the Cuckoo's Nest (1975),0.290601,76,264
Jurassic Park (1993),0.286174,120,261
Back to the Future (1985),0.285798,120,350
Dead Man Walking (1995),0.275720,59,299
Pulp Fiction (1994),0.262129,121,394
Terminator 2: Judgment Day (1991),0.261105,114,295


Overlap-adjusted recommendations:


,correlation,common_users,rating_count,score
Batman (1989),0.484477,121,201,0.342817
"Mask, The (1994)",0.480580,69,129,0.278655
Babe (1995),0.469349,72,219,0.276993
Indiana Jones and the Last Crusade (1989),0.344773,120,331,0.243369
"Lost World: Jurassic Park, The (1997)",0.444186,55,158,0.232669
Mrs. Doubtfire (1993),0.353906,91,192,0.228408
Batman Forever (1995),0.353607,90,114,0.227319
Michael (1996),0.577122,32,119,0.225218
Pretty Woman (1990),0.368376,74,164,0.219837
"Fish Called Wanda, A (1988)",0.329892,89,247,0.211226
